# Task 2 – Feature Engineering & Preprocessing
Input: `cleaned_data.csv` (Task 1) · Target: `target` (1 = satisfied). Stops before any model.

## 0. Load

In [14]:
import numpy as np
import pandas as pd, joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder



In [15]:
df = pd.read_csv("data/flight_satisfaction_data.csv")
print(df.shape)

(25976, 25)


## 01 – Feature Engineering

### 1.1 Identify features

In [16]:
TARGET = "target"
DROP   = ["id", "satisfaction"]        # id = identifier; satisfaction = text copy of the target (LEAKAGE)

cat_cols    = ["gender", "customer_type", "travel_type", "class"]
rating_cols = ["inflight_wifi_service", "departure_arrival_time_convenient", "ease_of_online_booking",
               "gate_location", "food_and_drink", "online_boarding", "seat_comfort", "inflight_entertainment",
               "on_board_service", "leg_room_service", "baggage_handling", "checkin_service",
               "inflight_service", "cleanliness"]
num_cols    = ["age", "flight_distance", "departure_delay_min", "arrival_delay_min"] + rating_cols

print(pd.crosstab(df["satisfaction"], df[TARGET]))   # proves 'satisfaction' duplicates the target

target                       0      1
satisfaction                         
neutral or dissatisfied  14573      0
satisfied                    0  11403


### 1.2 Create features (row-wise, no target)

In [17]:
def add_features(d):
    d = d.copy()
    rated = d[rating_cols].replace(0, np.nan)              # 0 = "not applicable"
    d["avg_rating"]    = rated.mean(axis=1)                # overall experience
    d["n_not_rated"]   = (d[rating_cols] == 0).sum(axis=1) # services not used
    d["delay_diff"]    = d["arrival_delay_min"] - d["departure_delay_min"]
    d["log_dep_delay"] = np.log1p(d["departure_delay_min"])# tames the skew
    return d

df = add_features(df)
new_cols = ["avg_rating", "n_not_rated", "delay_diff", "log_dep_delay"]
num_cols += new_cols
df[new_cols].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
avg_rating,25976.0,3.28,0.65,1.21,2.82,3.29,3.77,5.00
n_not_rated,25976.0,0.16,0.61,0.00,0.00,0.00,0.00,4.00
delay_diff,25976.0,0.51,9.94,-54.00,-2.00,0.00,0.00,124.00
log_dep_delay,25976.0,1.22,1.61,0.00,0.00,0.00,2.56,7.03


### 1.3 Review changes
| Action | Feature | Why |
|---|---|---|
| Added | `avg_rating` | One summary of the 14 service scores |
| Added | `n_not_rated` | Count of "not applicable" (0) ratings |
| Added | `delay_diff` | Did the flight gain/lose time in the air? |
| Added | `log_dep_delay` | Reduces the heavy right skew of delays |
| Removed | `id`, `satisfaction` | Identifier / target leakage |

## 02 – Data Splitting

In [18]:
X = df.drop(columns=[TARGET] + DROP)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)   # stratified: classes are 56/44

print(X_train.shape, X_test.shape)
print(round(y_train.mean(), 3), round(y_test.mean(), 3))

(20780, 26) (5196, 26)
0.439 0.439


Stratified 80/20 split. No date or group column, so a random split is enough. Everything below is **fitted on train only**.

## 03 – Preprocessing

In [19]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),   # 3.1 missing values
                      ("scale",  StandardScaler())]),                 # 3.3 scaling
     num_cols),
    ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore",  # 3.2 encoding
                          sparse_output=False),
     cat_cols),
]).set_output(transform="pandas")

X_train_p = preprocessor.fit_transform(X_train)   # learn on train
X_test_p  = preprocessor.transform(X_test)        # apply to test
print(X_train_p.shape, X_test_p.shape)

(20780, 28) (5196, 28)


## 04 – Final Checks

In [20]:
print(X_train_p.shape, X_test_p.shape)
display(X_train_p.head(3))

print("NaN train/test:", X_train_p.isna().sum().sum(), X_test_p.isna().sum().sum())
print("Inf train/test:", np.isinf(X_train_p).sum().sum(), np.isinf(X_test_p).sum().sum())
print("Same columns:", list(X_train_p.columns) == list(X_test_p.columns))
print("All numeric:", all(pd.api.types.is_numeric_dtype(t) for t in X_train_p.dtypes))

(20780, 28) (5196, 28)


,num__age,num__flight_distance,num__departure_delay_min,num__arrival_delay_min,num__inflight_wifi_service,num__departure_arrival_time_convenient,num__ease_of_online_booking,num__gate_location,num__food_and_drink,num__online_boarding,...,num__avg_rating,num__n_not_rated,num__delay_diff,num__log_dep_delay,cat__gender_Male,cat__customer_type_disloyal Customer,cat__travel_type_Personal Travel,cat__class_Business,cat__class_Eco,cat__class_Eco Plus
14748,-0.699109,-0.835410,-0.383067,-0.393866,-0.541340,0.621302,-0.537402,-0.765925,-0.914673,-0.926583,...,-0.873493,-0.265133,-0.052224,-0.761726,1.0,0.0,1.0,0.0,1.0,0.0
4297,0.686271,-0.569498,-0.142480,0.057191,0.959133,0.621302,1.585913,0.794766,-0.914673,1.282503,...,0.997372,-0.265133,0.759758,0.670387,0.0,0.0,0.0,1.0,0.0,0.0
14069,-0.237316,-1.091288,0.686207,0.428649,-1.291576,-0.031081,0.878141,0.014420,-0.162936,0.546141,...,-0.543340,-0.265133,-0.965704,1.547963,1.0,0.0,0.0,1.0,0.0,0.0


NaN train/test: 0 0
Inf train/test: 0 0
Same columns: True
All numeric: True


**Summary:** 4 new features, `id` and `satisfaction` (leakage) removed, stratified 80/20 split, median imputation + standardisation for numbers, one-hot for categories — all fitted on train only. Output is fully numeric with no missing values, ready for Task 3.